In [21]:
import oracledb
import time
import os
import json

def connect_db():
     script_dir = os.getcwd()
     config_path = os.path.join(script_dir, "db_config.json")
     with open(config_path, "r", encoding="utf-8") as cf:
          dbCfg = json.load(cf)
          
     try:
          connection = oracledb.connect(user=dbCfg["user"], password=dbCfg["pwd"], dsn=dbCfg["dsn"])
          print("Connection established successfully.")
          return connection
     except oracledb.DatabaseError as e:
          print(f"Database connection error: {e}")
          return None

In [22]:
try:
     with connect_db() as connection:
          with connection.cursor() as cursor:
               start_time = time.time()
               n = 1000
               result = cursor.callfunc("data_api.fget_ref_cursor", oracledb.CURSOR, [n])
               fetch_time = time.time() - start_time
               print(f"Function executed in {fetch_time:.4f} seconds.")

               row_count = 0
               #start_time = time.time()
               for row in result:
                    row_count += 1
               total_time = time.time() - start_time
               print(f"Fetched {row_count} rows in {total_time:.4f} seconds.")

except oracledb.DatabaseError as e:
     print(f"Error during database operation: {e}")


Connection established successfully.
Function executed in 0.0010 seconds.
Fetched 1000 rows in 0.0092 seconds.


In [ ]:
try:
     with connect_db() as connection:

          with connection.cursor() as cursor:

               start_time = time.perf_counter()
               # instead of cursor.var(oracledb.CURSOR) just create a new cursor to hold the REF CURSOR
               rc = connection.cursor()
               rows_needed = 10000
               cursor.callproc("data_api.get_ref_cursor", [rows_needed, rc])

               rows = rc.fetchall()
               rc.close()
               end_time = time.perf_counter()
               elapsed_time = end_time - start_time
               print(f"Time taken to fetch REF CURSOR data: {elapsed_time:.6f} seconds")
               print("Fetched rows from the REF CURSOR:")
               print("Number of rows:", len(rows))
               #for row in rows:
               #    print(row)
     
except oracledb.DatabaseError as e:
     print(f"Error connecting to the database: {e}")


Connection established successfully.
Time taken to fetch REF CURSOR data: 0.052399 seconds
Fetched rows from the REF CURSOR:
Number of rows: 10000


In [ ]:
try:
     with connect_db() as connection:
          with connection.cursor() as cursor:
     
               start_time = time.perf_counter()
               n = 10000
               rc = cursor.callfunc("data_api.fget_ref_cursor", oracledb.CURSOR, [n])

               rows = rc.fetchall()
               rc.close()
               end_time = time.perf_counter()
               elapsed_time = end_time - start_time
               print(f"Time taken to fetch REF CURSOR data: {elapsed_time:.6f} seconds")
               print("Number of rows:", len(rows))
               # for row in rows:
               #     print(row)

except oracledb.DatabaseError as e:
     print(f"Error connecting to the database: {e}")

Connection established successfully.
Time taken to fetch REF CURSOR data: 0.091451 seconds
Number of rows: 10000


In [19]:
try:
     with connect_db() as connection:
          with connection.cursor() as cursor:
     
               start_time = time.perf_counter()
               n = 10000
               json_data = cursor.callfunc("data_api.fget_json", oracledb.DB_TYPE_CLOB, [n])
               content = json_data.read()
               end_time = time.perf_counter()
               elapsed_time = end_time - start_time
               print(f"Time taken to fetch JSON data: {elapsed_time:.6f} seconds")
               print("Length of JSON data:", len(content))
               #print(content)
     
except oracledb.DatabaseError as e:
     print(f"Error connecting to the database: {e}")


Connection established successfully.
Time taken to fetch JSON data: 0.022556 seconds
Length of JSON data: 307789
